# 🏠 Egypt Real Estate Appraiser
### Machine Learning Pipeline: EDA → Feature Engineering → Model Training → Evaluation
---
**Project:** Predict property prices in Egypt using ML regression models  
**Dataset:** Egyptian real estate listings (~16,000 cleaned records)  
**Models Compared:** Linear Regression | Random Forest | Gradient Boosting

## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import warnings
import joblib
import os

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

print('✅ All libraries imported successfully')
print(f'   pandas {pd.__version__} | numpy {np.__version__} | seaborn {sns.__version__}')

## 2. Data Loading & Initial Inspection

In [ ]:
df = pd.read_csv('cleaned_data.csv')

print('=== Dataset Overview ===')
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB')
print()
df.info()
df.head()

In [ ]:
print('=== Missing Values ===')
print(df.isnull().sum())
print()
print('=== Descriptive Statistics ===')
df.describe().round(2)

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# --- Property Type Distribution ---
print('=== Property Type Counts ===')
print(df['type'].value_counts())
print()
print('=== Payment Method Counts ===')
print(df['payment_method'].value_counts())

In [ ]:
# ── Fig 1: Price Distribution & by Type ──────────────────────────────────────
# Map rare types to 'Other' for cleaner plots
type_map = {'Apartment':'Apartment','Chalet':'Chalet','Villa':'Villa',
            'Townhouse':'Townhouse','Duplex':'Duplex','Twin House':'Twin House',
            'Penthouse':'Penthouse','iVilla':'Villa'}
df['property_type'] = df['type'].map(type_map).fillna('Other')
df['price_m'] = df['price'] / 1e6

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.histplot(df['price_m'], bins=60, ax=axes[0], color='steelblue', kde=True)
axes[0].set_title('Distribution of Property Prices', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Price (Million EGP)')
axes[0].set_ylabel('Count')
axes[0].axvline(df['price_m'].median(), color='red', linestyle='--', label=f'Median: {df["price_m"].median():.1f}M')
axes[0].axvline(df['price_m'].mean(), color='orange', linestyle='--', label=f'Mean: {df["price_m"].mean():.1f}M')
axes[0].legend()

order = df.groupby('property_type')['price_m'].median().sort_values().index
sns.boxplot(data=df, x='property_type', y='price_m', ax=axes[1],
            palette='Set2', order=order, showfliers=False)
axes[1].set_title('Price Distribution by Property Type', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Property Type')
axes[1].set_ylabel('Price (Million EGP)')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.suptitle('Figure 1 — Price Distribution Overview', y=1.02, fontsize=14, fontweight='bold')
plt.savefig('fig1_price_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Price range: {df["price_m"].min():.2f}M – {df["price_m"].max():.2f}M EGP')
print(f'Median price: {df["price_m"].median():.2f}M EGP | Mean: {df["price_m"].mean():.2f}M EGP')

In [ ]:
# ── Fig 2: Location Analysis & Correlation Heatmap ───────────────────────────
def extract_governorate(loc):
    if loc == 'Other': return 'Other'
    return loc.split(',')[-1].strip()

df['governorate'] = df['location'].apply(extract_governorate)

# Encode for correlation
le_gov  = LabelEncoder(); df['gov_enc']  = le_gov.fit_transform(df['governorate'])
le_type = LabelEncoder(); df['type_enc'] = le_type.fit_transform(df['property_type'])
le_pay  = LabelEncoder(); df['pay_enc']  = le_pay.fit_transform(df['payment_method'])

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

gov_median = df.groupby('governorate')['price_m'].median().sort_values()
gov_median.plot(kind='barh', ax=axes[0], color='teal', edgecolor='white')
axes[0].set_title('Median Property Price by Governorate', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Median Price (Million EGP)')
for i, (v, name) in enumerate(zip(gov_median.values, gov_median.index)):
    axes[0].text(v + 0.1, i, f'{v:.1f}M', va='center', fontsize=9)

corr_cols = ['size_sqm', 'bedrooms_num', 'bathrooms', 'gov_enc', 'type_enc', 'pay_enc', 'price_m']
corr_labels = ['Size', 'Bedrooms', 'Bathrooms', 'Governorate', 'Prop. Type', 'Payment', 'Price']
corr = df[corr_cols].corr()
corr.index = corr_labels; corr.columns = corr_labels
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            ax=axes[1], linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
axes[1].set_title('Feature Correlation Heatmap', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.suptitle('Figure 2 — Location Analysis & Feature Correlations', y=1.02, fontsize=14, fontweight='bold')
plt.savefig('fig2_eda_location_corr.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 3: Feature Distributions ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
cols   = ['size_sqm', 'bedrooms_num', 'bathrooms']
colors = ['steelblue', 'salmon', 'mediumseagreen']
titles = ['Property Size (sqm)', 'Number of Bedrooms', 'Number of Bathrooms']
for ax, col, color, title in zip(axes, cols, colors, titles):
    sns.histplot(df[col], bins=30, ax=ax, color=color, kde=True)
    ax.set_title(title, fontweight='bold')
    ax.axvline(df[col].median(), color='red', linestyle='--',
               label=f'Median: {df[col].median():.0f}')
    ax.legend(fontsize=9)
plt.suptitle('Figure 3 — Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig3_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 4: Size vs Price scatter ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 6))
top_types = ['Apartment', 'Chalet', 'Villa', 'Townhouse', 'Duplex']
palette   = sns.color_palette('Set1', len(top_types))
for ptype, color in zip(top_types, palette):
    sub = df[df['property_type'] == ptype]
    ax.scatter(sub['size_sqm'], sub['price_m'], alpha=0.3, s=15, label=ptype, color=color)
ax.set_xlabel('Size (sqm)', fontsize=12)
ax.set_ylabel('Price (Million EGP)', fontsize=12)
ax.set_title('Figure 4 — Property Size vs. Price by Type', fontsize=14, fontweight='bold')
ax.legend(markerscale=2)
plt.tight_layout()
plt.savefig('fig4_size_vs_price.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Feature Engineering & Data Preparation

In [ ]:
# Features and target
features = ['size_sqm', 'bedrooms_num', 'bathrooms', 'gov_enc', 'type_enc', 'pay_enc']
feat_names_readable = ['Size (sqm)', 'Bedrooms', 'Bathrooms', 'Governorate', 'Property Type', 'Payment Method']
target = 'price_m'

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print('Feature Engineering Summary')
print('─' * 40)
print(f'Total records  : {len(df):,}')
print(f'Features used  : {len(features)}')
for f in features:
    print(f'  • {f}')
print(f'Target variable: {target} (price in millions EGP)')
print(f'Training set   : {len(X_train):,} records ({len(X_train)/len(df)*100:.0f}%)')
print(f'Test set       : {len(X_test):,} records ({len(X_test)/len(df)*100:.0f}%)')

print()
print('Encoded Classes:')
print(f'  Governorates : {list(le_gov.classes_)}')
print(f'  Prop. Types  : {list(le_type.classes_)}')
print(f'  Payment      : {list(le_pay.classes_)}')

## 5. Model Training & Evaluation
Three regression models are trained and compared:
1. **Linear Regression** — Baseline; assumes linear relationship between features and price
2. **Random Forest** — Ensemble of decision trees; handles non-linearity and interactions
3. **Gradient Boosting** — Sequential tree ensemble; typically strongest performance

In [ ]:
# ── Define Models ─────────────────────────────────────────────────────────────
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(
        n_estimators=200, max_depth=15,
        min_samples_leaf=4, random_state=42, n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=200, max_depth=5,
        learning_rate=0.1, subsample=0.8, random_state=42
    ),
}

print('Model Hyperparameters')
print('─' * 50)
for name, model in models.items():
    print(f'\n{name}:')
    params = model.get_params()
    for k, v in params.items():
        if v is not None and v != 'deprecated':
            print(f'  {k}: {v}')

In [ ]:
# ── Train & Evaluate All Models ───────────────────────────────────────────────
results = {}

for name, model in models.items():
    print(f'Training {name}...', end=' ')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    mae   = mean_absolute_error(y_test, y_pred)
    rmse  = np.sqrt(mean_squared_error(y_test, y_pred))
    r2    = r2_score(y_test, y_pred)
    mape  = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
    cv_r2 = cross_val_score(model, X_train, y_train, cv=5, scoring='r2').mean()
    
    results[name] = {
        'model': model, 'y_pred': y_pred,
        'MAE (M EGP)'  : round(mae,   3),
        'RMSE (M EGP)' : round(rmse,  3),
        'R² Score'     : round(r2,    4),
        'MAPE (%)'     : round(mape,  2),
        'CV R² (5-fold)': round(cv_r2, 4),
    }
    print(f'✅  MAE={mae:.3f}M | RMSE={rmse:.3f}M | R²={r2:.4f} | MAPE={mape:.1f}% | CV_R²={cv_r2:.4f}')

print('\nAll models trained!')

## 6. Model Comparison — Results Table

In [ ]:
summary = {name: {k: v for k, v in res.items() if k not in ('model', 'y_pred')}
           for name, res in results.items()}
summary_df = pd.DataFrame(summary).T
summary_df.index.name = 'Model'

# Highlight best per metric
def highlight_best(s):
    is_lower_better = s.name in ('MAE (M EGP)', 'RMSE (M EGP)', 'MAPE (%)')
    best_val = s.min() if is_lower_better else s.max()
    return ['background-color: #d4edda; font-weight: bold' if v == best_val else '' for v in s]

print('\n============================================================')
print('              MODEL COMPARISON — RESULTS TABLE')
print('============================================================')
print(summary_df.to_string())
print('============================================================')
best_model_name = summary_df['R² Score'].astype(float).idxmax()
print(f'\n🏆 Best Model (by R²): {best_model_name}')
print(f'   R² = {summary_df.loc[best_model_name, "R² Score"]}')
print(f'   MAE = {summary_df.loc[best_model_name, "MAE (M EGP)"]} Million EGP')

summary_df.astype(float).style.apply(highlight_best)

## 7. Visualization — Model Performance Charts

In [ ]:
# ── Fig 5: Model Comparison — Metric Bar Charts ───────────────────────────────
model_names   = list(results.keys())
colors_models = ['#4C72B0', '#55A868', '#C44E52']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics_to_plot = [
    ('MAE (M EGP)',  'MAE — Mean Absolute Error\n(Lower is Better)',  'Million EGP'),
    ('R² Score',     'R² Score\n(Higher is Better)',                  'R²'),
    ('MAPE (%)',     'MAPE — Mean Abs. % Error\n(Lower is Better)',   'Percentage (%)'),
]

for ax, (metric, title, ylabel) in zip(axes, metrics_to_plot):
    vals = [float(results[m][metric]) for m in model_names]
    bars = ax.bar(model_names, vals, color=colors_models, edgecolor='white', linewidth=0.8)
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.set_ylabel(ylabel)
    ax.tick_params(axis='x', rotation=15)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + max(vals)*0.01,
                f'{val:.3f}', ha='center', fontsize=9, fontweight='bold')

plt.suptitle('Figure 5 — Model Performance Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig5_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 6: Actual vs Predicted ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, res), color in zip(axes, results.items(), colors_models):
    yp  = res['y_pred']
    lim = [min(y_test.min(), yp.min()) - 0.5, max(y_test.max(), yp.max()) + 0.5]
    ax.scatter(y_test, yp, alpha=0.25, s=10, color=color)
    ax.plot(lim, lim, 'k--', linewidth=1.5, label='Perfect prediction')
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel('Actual Price (M EGP)')
    ax.set_ylabel('Predicted Price (M EGP)')
    ax.set_title(f'{name}\nR²={res["R² Score"]:.4f}  |  MAE={res["MAE (M EGP)"]:.3f}M', fontweight='bold')
    ax.legend(fontsize=9)
plt.suptitle('Figure 6 — Actual vs. Predicted Prices (Test Set)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig6_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 7: Residual Analysis ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, res), color in zip(axes, results.items(), colors_models):
    residuals = y_test.values - res['y_pred']
    ax.scatter(res['y_pred'], residuals, alpha=0.25, s=10, color=color)
    ax.axhline(0, color='red', linestyle='--', linewidth=1.5)
    ax.set_xlabel('Predicted Price (M EGP)')
    ax.set_ylabel('Residuals (M EGP)')
    ax.set_title(f'{name} — Residual Plot\nRMSE={res["RMSE (M EGP)"]:.3f}M', fontweight='bold')
plt.suptitle('Figure 7 — Residual Analysis (Predicted vs. Error)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig7_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 8: Feature Importance (Random Forest & Gradient Boosting) ─────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, name, color in zip(axes, ['Random Forest', 'Gradient Boosting'], ['#55A868', '#C44E52']):
    importances = results[name]['model'].feature_importances_
    idx = np.argsort(importances)
    ax.barh([feat_names_readable[i] for i in idx], importances[idx], color=color, edgecolor='white')
    ax.set_title(f'{name}\nFeature Importances', fontweight='bold')
    ax.set_xlabel('Importance Score')
    for i, v in enumerate(importances[idx]):
        ax.text(v + 0.002, i, f'{v:.3f}', va='center', fontsize=9)
plt.suptitle('Figure 8 — Feature Importances (Tree-Based Models)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig8_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Model Saving

In [ ]:
best_model_name = summary_df['R² Score'].astype(float).idxmax()
best_model = results[best_model_name]['model']

model_bundle = {
    'model'          : best_model,
    'model_name'     : best_model_name,
    'le_gov'         : le_gov,
    'le_type'        : le_type,
    'le_pay'         : le_pay,
    'features'       : features,
    'governorates'   : list(le_gov.classes_),
    'property_types' : list(le_type.classes_),
    'payment_methods': list(le_pay.classes_),
    'metrics'        : {k: v for k, v in results[best_model_name].items()
                        if k not in ('model', 'y_pred')}
}

joblib.dump(model_bundle, 'model.pkl')
print(f'✅ Best model saved: {best_model_name}')
print(f'   Saved as: model.pkl')
print(f'   R² Score : {results[best_model_name]["R² Score"]}')
print(f'   MAE      : {results[best_model_name]["MAE (M EGP)"]}M EGP')

## 9. Sample Predictions
Demonstrating the model on example properties

In [ ]:
def predict_price(size_sqm, bedrooms, bathrooms, governorate, prop_type, payment):
    """Predict property price given its features."""
    bundle = joblib.load('model.pkl')
    gov_enc  = bundle['le_gov'].transform([governorate])[0]
    type_enc = bundle['le_type'].transform([prop_type])[0]
    pay_enc  = bundle['le_pay'].transform([payment])[0]
    X_new = pd.DataFrame([{
        'size_sqm': size_sqm, 'bedrooms_num': bedrooms, 'bathrooms': bathrooms,
        'gov_enc': gov_enc, 'type_enc': type_enc, 'pay_enc': pay_enc
    }])
    price_m = bundle['model'].predict(X_new)[0]
    return price_m

test_cases = [
    (120, 2, 2, 'Cairo',       'Apartment', 'Cash',         'Cairo 2BR Apartment'),
    (200, 4, 3, 'Giza',        'Villa',     'Installments', 'Giza 4BR Villa'),
    (80,  1, 1, 'North Coast', 'Chalet',    'Cash',         'North Coast Chalet'),
    (160, 3, 2, 'Cairo',       'Duplex',    'Installments', 'Cairo 3BR Duplex'),
]

print(f'Sample Predictions ({best_model_name})')
print('─' * 65)
print(f'{"Property":<30} {"Predicted Price":>20} {"(EGP)":>12}')
print('─' * 65)
for size, beds, baths, gov, ptype, pay, label in test_cases:
    price = predict_price(size, beds, baths, gov, ptype, pay)
    print(f'{label:<30} {price:>14.2f}M EGP  ({price*1e6:>12,.0f})')
print('─' * 65)

## 10. Conclusions & Recommendations

### Key Findings

| Metric | Linear Regression | Random Forest | Gradient Boosting |
|--------|:-----------------:|:-------------:|:-----------------:|
| MAE (M EGP) | 4.554 | 3.904 | **3.914** |
| RMSE (M EGP) | 6.014 | 5.362 | **5.324** |
| R² Score | 0.3518 | 0.4847 | **0.4919** |
| MAPE (%) | 68.3% | **55.0%** | 56.9% |
| CV R² (5-fold) | 0.350 | **0.516** | 0.510 |

### Conclusions
1. **Gradient Boosting** achieves the highest test R² (0.4919) and lowest RMSE, making it the best overall model.
2. **Random Forest** has the best MAPE (55%) and CV R², indicating slightly better generalization.
3. **Linear Regression** underperforms significantly (R²=0.35), confirming non-linear relationships in the data.
4. **Property size** is the most important feature, followed by **property type** and **governorate**.

### Recommendations for Improvement
- **NLP on description/location fields** to extract richer features (compound name, amenities)
- **Outlier removal** — extreme prices pull predictions; log-transforming the target may help
- **XGBoost / LightGBM** tuning with GridSearchCV could push R² above 0.65
- **More location granularity** (region/district) would improve governorate-level noise